
# Proyecto 2 — PageRank sobre redes reales
### IMT2230 — Álgebra Lineal Avanzada y Modelamiento, PUC

**Red elegida:** *Wikipedia links (csb)* — KONECT (http://konect.cc/networks/wikipedia_link_csb/)

**Grupo:** _Bastián Pérez, Martín Pérez, Felipe Serrano_


> Nota de reproducibilidad: la celda de código siguiente descarga y
> descomprime automáticamente los datos desde KONECT (usando `urllib`). No es
> necesario descargar nada a mano, basta con tener conexión a internet
> la primera vez que se ejecuta el notebook.


In [5]:

import os
import urllib.request
import tarfile

URL = "http://konect.cc/files/download.tsv.wikipedia_link_csb.tar.bz2"
CARPETA_DATOS = "wikipedia_link_csb"
ARCHIVO_DESCARGA = "download.tsv.wikipedia_link_csb.tar.bz2"

# Descarga automática desde KONECT solo si los datos no están ya presentes
# localmente.
if not os.path.isdir(CARPETA_DATOS):
    print(f"Descargando datos desde {URL} ...")
    urllib.request.urlretrieve(URL, ARCHIVO_DESCARGA)
    print("Descomprimiendo...")
    with tarfile.open(ARCHIVO_DESCARGA, "r:bz2") as tar:
        tar.extractall(".")
    print("Listo.")
else:
    print(f"Carpeta '{CARPETA_DATOS}' ya existe, se omite la descarga.")


Carpeta 'wikipedia_link_csb' ya existe, se omite la descarga.


In [6]:

# Librerías utilizadas en todo el notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from scipy import sparse, stats

np.random.seed(42)


ModuleNotFoundError: No module named 'matplotlib'


## P1 — Elección y descripción de la red

**Nodos:** artículos de la Wikipedia en **kashubio** (lengua eslava
minoritaria del norte de Polonia, código ISO `csb`).

**Aristas dirigidas:** $u \to v$ si el artículo $u$ contiene un
hipervínculo (`wikilink`) hacia el artículo $v$.

**Fuente y contexto:** KONECT extrajo la red directamente de un dump de
`dumps.wikimedia.org`. Es una red de hipervínculos web clásica.

**Por qué la elegimos:** cumple los mínimos (≥500 nodos,
≥2000 aristas, dirigida), y al ser una Wikipedia pequeña, es más fácil de visualizar.

**Limitación a tener en cuenta:** este dataset de KONECT **no** trae
nombres de artículo, solo IDs internos de Wikipedia.


In [ ]:

RUTA = "wikipedia_link_csb/out.wikipedia_link_csb"

# ---- Carga de datos --------------------------------------------------
# Cada línea "u v" es una arista dirigida u -> v (u enlaza a v).
# La primera línea "% asym unweighted" es un comentario y se descarta.
edges = pd.read_csv(RUTA, sep=r"\s+", comment="%", header=None,
                     names=["from", "to"], dtype=np.int64)
print(f"Aristas leídas (antes de limpieza): {len(edges)}")

# ---- Limpieza: duplicados exactos y auto-loops (u -> u) ----------------
# Un auto-loop no aporta información real de navegación entre artículos
# distintos, y un duplicado exacto no debería contar dos veces como
# "peso de salida" en H.
n_dup_antes = len(edges)
edges = edges.drop_duplicates()
n_dup = n_dup_antes - len(edges)

n_self_antes = len(edges)
edges = edges[edges["from"] != edges["to"]].reset_index(drop=True)
n_self = n_self_antes - len(edges)

print(f"Duplicados eliminados: {n_dup} | Auto-loops eliminados: {n_self}")
print(f"Aristas finales (m): {len(edges)}")

# ---- Reindexado de nodos a 0..n-1 --------------------------------------
# Los IDs originales de Wikipedia no son consecutivos ni parten en 0.
nodos_originales = np.union1d(edges["from"].unique(), edges["to"].unique())
n = len(nodos_originales)
id_to_idx = {node_id: i for i, node_id in enumerate(nodos_originales)}
edges["u"] = edges["from"].map(id_to_idx)  # índice interno del nodo origen
edges["v"] = edges["to"].map(id_to_idx)    # índice interno del nodo destino
m = len(edges)

# ---- Grados de entrada y salida ----------------------------------------
out_deg = np.zeros(n, dtype=np.int64)
in_deg = np.zeros(n, dtype=np.int64)
np.add.at(out_deg, edges["u"].to_numpy(), 1)
np.add.at(in_deg, edges["v"].to_numpy(), 1)

nodo_max_in_idx = int(np.argmax(in_deg))
densidad = m / (n * (n - 1))
nodos_colgantes = int(np.sum(out_deg == 0))

resumen = pd.DataFrame({
    "Estadística": [
        "Número de nodos (n)", "Número de aristas (m)",
        "Grado de entrada medio (d̄_in)", "Grado de salida medio (d̄_out)",
        "Nodo de mayor grado de entrada (ID original)",
        "Máximo grado de entrada", "Densidad m / (n(n-1))",
        "Nodos colgantes (out-degree = 0)", "Fracción de nodos colgantes",
    ],
    "Valor": [
        n, m, round(in_deg.mean(), 4), round(out_deg.mean(), 4),
        int(nodos_originales[nodo_max_in_idx]), int(in_deg[nodo_max_in_idx]),
        f"{densidad:.6f}", nodos_colgantes, f"{nodos_colgantes/n:.4%}",
    ]
})
resumen


: 


## P2 — Pregunta e hipótesis inicial

**Pregunta.** ¿Qué artículos son los más influyentes o centrales dentro
de la Wikipedia en kashubio, entendiendo "influyente" como el artículo
hacia el cual convergería con mayor frecuencia un lector que navegara la
enciclopedia siguiendo enlaces al azar durante un tiempo prolongado?

**Hipótesis inicial**.  Esperamos que los nodos con mayor
grado de entrada concentren también el mayor PageRank, ya que en redes de
hipervínculos wiki los artículos "hub" (por ejemplo países, años o
entidades administrativas muy referenciadas) suelen ser enlazados desde
gran cantidad de artículos distintos. Sin embargo, dado que la red tiene
densidad baja (0.62%) y nodos colgantes, anticipamos que **PageRank y
grado de entrada no coincidirán perfectamente**: deberían aparecer
artículos con in-degree moderado pero enlazados desde otros nodos ya
influyentes.



## P3 — Análisis exploratorio de la red

Antes de calcular PageRank, inspeccionamos la distribución de grados, los
nodos colgantes, los nodos de mayor grado y la conectividad de la red.


In [ ]:

# (a) Distribución del grado de entrada y de salida, en escala log-log:
# las redes de hipervínculos suelen mostrar distribuciones con muchos 
# nodos de grado bajo y pocos hubs de grado muy alto,
# que un histograma lineal común aplastaría visualmente.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, deg, titulo, color in [
    (axes[0], in_deg, "Grado de entrada (in-degree)", "#1f77b4"),
    (axes[1], out_deg, "Grado de salida (out-degree)", "#d62728"),
]:
    deg_pos = deg[deg > 0]  # log(0) no está definido
    valores, conteos = np.unique(deg_pos, return_counts=True)
    ax.scatter(valores, conteos, s=14, color=color, alpha=0.7)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("Grado (escala log)"); ax.set_ylabel("Cantidad de nodos (escala log)")
    ax.set_title(titulo); ax.grid(True, which="both", alpha=0.3)
fig.suptitle("Distribución del grado de entrada y de salida — Wikipedia (csb)")
fig.tight_layout()
plt.show()


In [ ]:

# (b) Nodos colgantes (out-degree = 0): problemáticos porque generan
# columnas de ceros en H, rompiendo la propiedad columna-estocástica.
# Se identifican comparando out_deg == 0, y se resuelven
# reemplazando esas columnas por el vector uniforme 1/n en S (modela
# "teletransportación" cuando el lector cae en un artículo sin salidas).
print(f"Nodos colgantes: {nodos_colgantes} ({nodos_colgantes/n:.4%} del total)")


NameError: name 'nodos_colgantes' is not defined

In [ ]:

# (c) Top-10 nodos por grado de entrada y por grado de salida.
# (Reportamos el ID original de Wikipedia, no el índice interno.)
top10_in_idx = np.argsort(-in_deg)[:10]
top10_out_idx = np.argsort(-out_deg)[:10]

top10_in = pd.DataFrame({"Rango": range(1, 11),
                          "ID original": nodos_originales[top10_in_idx],
                          "Grado de entrada": in_deg[top10_in_idx]})
top10_out = pd.DataFrame({"Rango": range(1, 11),
                           "ID original": nodos_originales[top10_out_idx],
                           "Grado de salida": out_deg[top10_out_idx]})

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(top10_in["ID original"].astype(str), top10_in["Grado de entrada"], color="#1f77b4")
axes[0].set_title("Top-10 grado de entrada"); axes[0].tick_params(axis="x", rotation=45)
axes[1].bar(top10_out["ID original"].astype(str), top10_out["Grado de salida"], color="#d62728")
axes[1].set_title("Top-10 grado de salida"); axes[1].tick_params(axis="x", rotation=45)
fig.suptitle("Nodos con mayor grado — Wikipedia (csb)")
fig.tight_layout()
plt.show()

top10_in


In [ ]:

# (d) Conectividad: ¿la red es fuertemente conexa? ¿hay componentes aisladas?
G = nx.DiGraph()
G.add_nodes_from(range(n))
G.add_edges_from(edges[["u", "v"]].itertuples(index=False, name=None))

es_fuertemente_conexa = nx.is_strongly_connected(G)
scc_sizes = sorted((len(c) for c in nx.strongly_connected_components(G)), reverse=True)
wcc_sizes = sorted((len(c) for c in nx.weakly_connected_components(G)), reverse=True)

print(f"¿Es fuertemente conexa? {es_fuertemente_conexa}")
print(f"Nº de componentes fuertemente conexas: {len(scc_sizes)} | top 5 tamaños: {scc_sizes[:5]}")
print(f"Nº de componentes débilmente conexas: {len(wcc_sizes)} | top 5 tamaños: {wcc_sizes[:5]}")

# Guardamos la SCC gigante para usarla más adelante
scc_gigante = max(nx.strongly_connected_components(G), key=len)
en_scc_gigante = np.isin(np.arange(n), list(scc_gigante))



La red **no** es fuertemente conexa (hay miles de SCC
pequeñas y una gigante de ~48% de los nodos), pero esto **no es un
problema** para PageRank: como se construye en P4, la Matriz de Google
$G$ es siempre irreducible (todas sus entradas son estrictamente
positivas gracias al término de teletransporte), por lo que $G$ tiene una
distribución estacionaria única sin importar la conectividad del grafo
subyacente.



## P4 — Construcción de la Matriz de Google

Notación:

$$H_{ij} = \begin{cases} 1/\text{out}(j) & \text{si } j \to i \\ 0 & \text{si no} \end{cases}
\qquad
S = H + \frac{1}{n}\,\mathbf{a}\,\mathbf{1}^T
\qquad
G = \alpha S + \frac{1-\alpha}{n}\,\mathbf{1}\mathbf{1}^T$$

Con $n=5561$, una matriz $G$ densa tendría ~31 millones de entradas
(247 MB en float64) — manejable, pero la pauta pide "implementación
eficiente de la multiplicación matriz-vector". Por eso construimos $H$
como matriz **dispersa** (`scipy.sparse`) y aplicamos $S$ y $G$ a un
vector usando la identidad algebraica de rank-1, sin materializar
matrices densas completas (esto se usa recién en P5).


In [ ]:

ALPHA = 0.85  # factor de amortiguamiento estándar

# (a) Matriz de hipervínculos H, dispersa: H_ij = 1/out(j) si j -> i.
u_arr = edges["u"].to_numpy()  # nodo origen j
v_arr = edges["v"].to_numpy()  # nodo destino i
valores_H = 1.0 / out_deg[u_arr]
H = sparse.coo_matrix((valores_H, (v_arr, u_arr)), shape=(n, n)).tocsr()

# Verificación: columnas no colgantes suman 1, columnas colgantes suman 0.
suma_columnas_H = np.asarray(H.sum(axis=0)).flatten()
es_colgante = (out_deg == 0)
print("Columnas no colgantes suman 1:", np.allclose(suma_columnas_H[~es_colgante], 1.0))
print("Columnas colgantes suman 0:   ", np.allclose(suma_columnas_H[es_colgante], 0.0))
print(f"Columnas cero (nodos colgantes): {int(es_colgante.sum())} "
      f"({es_colgante.mean():.4%} del total)")



**Por qué cada columna de $H$ suma 1** (si el nodo fuente no es
colgante): el nodo $j$ reparte su "peso" de salida en partes iguales
($1/\text{out}(j)$) entre sus $\text{out}(j)$ enlaces, así que la columna
$j$ distribuye exactamente una unidad de probabilidad entre sus vecinos.
Las columnas cero se identifican exactamente comparando
`out_deg == 0`.


In [ ]:

# (b) Matriz columna-estocástica S = H + (1/n) a 1^T.
# No materializamos S densa: guardamos el vector indicador de colgantes
# 'a' (a_j = 1 si el nodo j es colgante) y aplicamos la corrección
# directamente sobre vectores en P5.
a = es_colgante.astype(np.float64)

# Verificación conceptual sobre columnas colgantes de muestra:
# columna colgante j: suma de S = 0 (de H) + a_j * n * (1/n) = 1
for j in np.where(es_colgante)[0][:3]:
    suma_col_S = suma_columnas_H[j] + a[j] * n * (1.0 / n)
    print(f"columna colgante j={j}: suma de S = {suma_col_S:.6f}")



**(c) Matriz de Google $G$.** Justificación de $\alpha=0.85$: es el valor
clásico que se usa; balancea
fidelidad a la estructura real de enlaces ($S$) con la garantía teórica
de ergodicidad.

**Por qué todas las entradas de $G$ son estrictamente positivas:**
$G_{ij} = \alpha S_{ij} + \frac{1-\alpha}{n}$. Como $S_{ij}\ge 0$ siempre
(es una probabilidad) y $\frac{1-\alpha}{n}>0$ estrictamente (porque
$\alpha=0.85<1$), la suma es estrictamente positiva para **toda** entrada
$(i,j)$, sin importar la estructura del grafo original. Esto es lo que
garantiza que $G$ sea irreducible y aperiódica (ergódica).


In [ ]:

print(f"alpha = {ALPHA}")
print(f"(1-alpha)/n = {(1-ALPHA)/n:.3e} > 0  =>  G_ij > 0 para toda entrada (i,j)")


## P5 — Cálculo del PageRank mediante iteración de potencias

Reutilizamos el patrón visto para la convención columna-estocástica:
$\pi_{t+1} = P\,\pi_t$. Aquí $P=G$, pero aplicamos $G$ a un vector sin
materializarla densa, usando la siguiente identidad algebraica.

Partiendo de $G = \alpha S + \frac{1-\alpha}{n}\mathbf{1}\mathbf{1}^T$ y
$S = H + \frac{1}{n}\mathbf{a}\mathbf{1}^T$, y notando que
$\mathbf{1}^T\mathbf{r} = \sum_i r_i$ (la suma de las entradas de
$\mathbf{r}$), se obtiene:

$$G\mathbf{r} = \alpha (H\mathbf{r}) + \frac{\alpha}{n}\Big(\sum_j a_j r_j\Big) + \frac{1-\alpha}{n}\Big(\sum_i r_i\Big)$$

Aquí es importante notar que el vector $\mathbf{a}$ (indicador de nodos
colgantes) está indexado por **columna** $j$, no por fila $i$: en
$S=H+\frac{1}{n}\mathbf{a}\mathbf{1}^T$ se tiene
$(\mathbf{a}\mathbf{1}^T)_{ij}=a_j$, es decir, un valor constante en $i$
para cada columna $j$. Por eso la corrección por nodos colgantes es un
**escalar** — la masa total de $\mathbf{r}$ que está parada en nodos
colgantes, $\sum_j a_j r_j$ — que se reparte por igual entre **todas**
las filas del vector resultante, en vez de multiplicarse componente a
componente contra $\mathbf{a}$.

In [ ]:
def aplicar_G(r):
    """Calcula G @ r sin materializar G de forma densa."""
    suma_r = r.sum()
    masa_colgante = np.dot(a, r)  # sum_j a_j * r_j: masa de r en nodos colgantes
    return ALPHA * (H @ r) + ALPHA * masa_colgante / n + (1 - ALPHA) * suma_r / n

# (a) Iteración de potencias desde r^(0) = 1/n, hasta convergencia.
EPS = 1e-10
MAX_ITER = 1000
r = np.full(n, 1.0 / n)
historial = [r.copy()]  # guardamos cada r^(k) para poder medir la distancia a r*
k = 0
while True:
    r_nuevo = aplicar_G(r)
    err_consecutivo = np.sum(np.abs(r_nuevo - r))
    r = r_nuevo
    historial.append(r.copy())
    k += 1
    if err_consecutivo < EPS or k >= MAX_ITER:
        break

historial = np.array(historial)
r_estrella = historial[-1]
print(f"Iteraciones necesarias para convergencia (eps={EPS:.0e}): {k}")

In [ ]:
# (b) Curva de convergencia: error ||r^(k) - r*||_1 en función de k, escala log.
# Usamos el r* ya convergido (última fila de 'historial') como referencia, y
# medimos qué tan lejos estaba cada r^(k) intermedio de ese límite.
errores = np.sum(np.abs(historial[:-1] - r_estrella), axis=1)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.semilogy(range(len(errores)), errores, marker="o", markersize=2, lw=1)
ax.set_xlabel("Iteración k")
ax.set_ylabel(r"$\|\mathbf{r}^{(k)} - \mathbf{r}^*\|_1$ (escala log)")
ax.set_title("Curva de convergencia de la iteración de potencias")
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
plt.show()

# Razón geométrica aproximada de decaimiento (régimen ya estable):
razones = errores[1:] / errores[:-1]
razon_estable = np.median(razones[len(razones)//2:-1])
print(f"Razón de decaimiento (mediana, 2ª mitad): {razon_estable:.4f}  (alpha={ALPHA})")

In [ ]:

# (c) Verificación de propiedades teóricas.
norma_1 = np.sum(np.abs(r_estrella))
todos_positivos = np.all(r_estrella > 0)
print(f"||r*||_1 = {norma_1:.10f}  (debe ser 1)")
print(f"¿Todo r*_i > 0? {todos_positivos}  (mínimo = {r_estrella.min():.3e})")



**Por qué se garantizan:** $G$ es columna-estocástica (cada columna suma
1), así que conserva la norma 1 en cada iteración: como
$\mathbf{r}^{(0)}$ tiene suma 1, $\|\mathbf{r}^{(k)}\|_1=1$ para todo $k$,
incluido el límite $\mathbf{r}^*$. La positividad estricta viene de que
$G$ tiene **todas** sus entradas $>0$ (P4): $\mathbf{r}^*=G\mathbf{r}^*$
es una combinación positiva de un vector no negativo y no nulo, por lo
tanto $r^*_i>0$ para todo $i$.


In [ ]:

# (d) Tabla de los 20 nodos de mayor PageRank.
top20_idx = np.argsort(-r_estrella)[:20]
tabla_top20 = pd.DataFrame({
    "Rango": range(1, 21),
    "ID original": nodos_originales[top20_idx],
    "PageRank (r*)": r_estrella[top20_idx],
    "Grado de entrada": in_deg[top20_idx],
    "Grado de salida": out_deg[top20_idx],
})
tabla_top20



## P6 — PageRank vs. grado de entrada


In [ ]:

# (a) Scatter in-degree vs PageRank (log-log) + correlación de Pearson.
top10_idx = np.argsort(-r_estrella)[:10]

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.scatter(in_deg, r_estrella, s=10, alpha=0.35, color="#7f7f7f", label="Todos los nodos")
ax.scatter(in_deg[top10_idx], r_estrella[top10_idx], s=60, color="#d62728",
           edgecolor="black", zorder=3, label="Top-10 PageRank")
for idx in top10_idx:
    ax.annotate(str(nodos_originales[idx]), (in_deg[idx], r_estrella[idx]),
                fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("Grado de entrada (in-degree)"); ax.set_ylabel("PageRank (r*)")
ax.set_title("PageRank vs. grado de entrada — Wikipedia (csb)")
ax.set_xscale("log"); ax.set_yscale("log"); ax.legend()
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
plt.show()

pearson_r, pearson_p = stats.pearsonr(in_deg, r_estrella)
print(f"Correlación de Pearson (in-degree, PageRank): {pearson_r:.4f} (p={pearson_p:.2e})")


In [ ]:

# (b) Nodos donde PageRank e in-degree difieren significativamente,
# comparando por RANGO (posición en el ranking) en vez de valor crudo.
rango_in = pd.Series(-in_deg).rank(method="min").to_numpy()
rango_pr = pd.Series(-r_estrella).rank(method="min").to_numpy()
diferencia = rango_in - rango_pr  # positivo => mejor en PageRank que en in-degree

df_cmp = pd.DataFrame({
    "ID original": nodos_originales, "in_degree": in_deg,
    "rango_in_degree": rango_in.astype(int), "PageRank": r_estrella,
    "rango_PageRank": rango_pr.astype(int), "diferencia_rango": diferencia.astype(int),
})

# Alto in-degree (top-50), pero PageRank relativamente bajo:
print("=== Alto in-degree, PageRank relativamente bajo ===")
print(df_cmp[df_cmp["rango_in_degree"] <= 50].sort_values("diferencia_rango").head(5).to_string(index=False))

# PageRank alto (top-50), pero in-degree moderado:
print("\n=== PageRank alto, in-degree moderado ===")
print(df_cmp[df_cmp["rango_PageRank"] <= 50].sort_values("diferencia_rango", ascending=False).head(5).to_string(index=False))



La correlación de Pearson (≈0.38) es positiva pero moderada: el
in-degree explica solo parte de la importancia real. Encontramos nodos
con in-degree alto (424, rango 49) que caen hasta el rango ~900 en
PageRank —reciben muchos enlaces, pero probablemente desde artículos
poco relevantes—, y el caso opuesto más extremo: un nodo con solo 17
enlaces entrantes (rango 611) que alcanza el top-48 de PageRank,
heredando importancia concentrada desde muy pocas fuentes de alto valor.
Esto es exactamente el fenómeno que PageRank captura y que un conteo
simple de enlaces no puede ver.



## P7 — Interpretación de resultados

Este dataset de KONECT no trae
atributos naturales de nodo (sin títulos de artículo ni categoría
temática). Como "atributo" disponible para colorear el subgrafo usamos la
pertenencia a la componente fuertemente conexa (SCC) gigante calculada en
P3(d): es un atributo estructural real de la red, no inventado.


In [ ]:

# (b) Subgrafo inducido por los 40 nodos de mayor PageRank.
N_TOP = 40
top_idx = np.argsort(-r_estrella)[:N_TOP]
top_set = set(top_idx.tolist())
edges_sub = edges[edges["u"].isin(top_set) & edges["v"].isin(top_set)]

Gs = nx.DiGraph()
Gs.add_nodes_from(top_idx.tolist())
Gs.add_edges_from(edges_sub[["u", "v"]].itertuples(index=False, name=None))
pos = nx.spring_layout(Gs, seed=42, k=0.9)

tamanos = np.array([r_estrella[i] for i in Gs.nodes()])
tamanos_escalados = 800 + 25000 * (tamanos - tamanos.min()) / (tamanos.max() - tamanos.min())
colores = ["#2ca02c" if en_scc_gigante[i] else "#d62728" for i in Gs.nodes()]
etiquetas = {i: str(nodos_originales[i]) for i in Gs.nodes()}

fig, ax = plt.subplots(figsize=(10, 8))
nx.draw_networkx_edges(Gs, pos, ax=ax, alpha=0.3, arrows=True, arrowsize=8,
                        connectionstyle="arc3,rad=0.05")
nx.draw_networkx_nodes(Gs, pos, ax=ax, node_size=tamanos_escalados,
                        node_color=colores, edgecolors="black", linewidths=0.8)
nx.draw_networkx_labels(Gs, pos, labels=etiquetas, ax=ax, font_size=7)
leyenda = [
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#2ca02c",
               markersize=10, label="En la SCC gigante (núcleo navegable)"),
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#d62728",
               markersize=10, label="Fuera de la SCC gigante"),
]
ax.legend(handles=leyenda, loc="upper left", fontsize=9)
ax.set_title(f"Subgrafo inducido por los {N_TOP} nodos de mayor PageRank\n(tamaño ∝ PageRank)")
ax.axis("off")
fig.tight_layout()
plt.show()

n_en_scc = int(en_scc_gigante[top_idx].sum())
dens_sub = nx.density(Gs)
print(f"{n_en_scc}/{N_TOP} nodos del top-PageRank están en la SCC gigante "
      f"({n_en_scc/N_TOP:.1%}), vs. {len(scc_gigante)/n:.1%} de la red completa.")
print(f"Densidad del subgrafo top-{N_TOP}: {dens_sub:.4f} vs. densidad global {m/(n*(n-1)):.6f}")



**(a) Por qué tienen sentido como nodos influyentes.** Los nodos 516 y
2646 combinan in-degree muy alto **y** out-degree moderado-alto, es
decir, funcionan como hubs bidireccionales: reciben de muchos artículos y
a la vez enlazan hacia otros nodos importantes, propagando su propio
peso recursivamente (la esencia de PageRank frente a un conteo simple).

**(b) Estructura observada.** El 100% de los 40 nodos de mayor PageRank
está dentro de la SCC gigante (que cubre solo el 48% de la red completa):
ningún nodo periférico se cuela entre los influyentes. La densidad del
subgrafo top-40 es ~30 veces la densidad global: los nodos influyentes
están mucho más interconectados entre sí que el resto de la red.

**(c) Atributos naturales.** Esta red no trae etiquetas temáticas (a
diferencia, por ejemplo, de una red de blogs políticos), por lo que no
podemos evaluar si el top-PageRank pertenece a "un grupo particular" en
sentido semántico — limitación real de los datos, documentada aquí en
vez de asumida.



## P8 — Discusión, limitaciones y conclusiones

**¿Se cumplió la hipótesis?** Parcialmente. Acertamos en que los nodos de
mayor in-degree (516, 2646) también lideran el PageRank, pero la
correlación de solo 0.38 y los casos de P6 muestran que **la estructura
recursiva domina sobre el conteo bruto de enlaces**, tal como anticipaba
la segunda parte de la hipótesis.

**Limitaciones del método y de los datos.**
- El modelo de marcha aleatoria tiene sentido: es literalmente cómo
  navega un lector siguiendo wikilinks.
- Los 99 nodos colgantes (1.8%) son pocos y quedan bien manejados por
  $S$; no distorsionan de forma importante el resultado.
- La red es dispersa (densidad 0.62%), régimen normal para redes de
  hipervínculos donde PageRank es informativo (en una red casi completa
  todos tendrían PageRank similar)
- Limitación real más importante: **sin nombres de artículo no podemos
  validar semánticamente** qué son los nodos 516 o 2646 — el análisis es
  puramente estructural.

**Preguntas nuevas.** El hallazgo más claro que no anticipamos es que el
núcleo de los 40 nodos con mayor PageRank resultó ser ~30 veces más denso
que la red completa, esto abre la pregunta de **por qué los nodos
influyentes están tan concentrados entre sí**, algo que no exploramos más
allá de constatarlo. También queda abierta la pregunta de **qué hace que
un nodo con in-degree alto reciba PageRank bajo**: ¿viene
principalmente de otros nodos con PageRank bajo? No pudimos profundizar en
esto porque el dataset no incluye los títulos de los artículos, así que no
hay forma de inspeccionar el contenido real de esos nodos "discordantes".

**Explicación no técnica.** Un puñado de artículos de la Wikipedia en
kashubio actúan como "estaciones centrales" de la enciclopedia — no
simplemente porque muchos artículos los mencionen, sino porque los
artículos que sí los mencionan son a su vez importantes. Es como ser
recomendado por gente influyente en vez de por muchos desconocidos:
cuenta más *quién* te enlaza que *cuántos* te enlazan.